In [4]:
start_patterns = [
    r"(?i)^1\.\s*Project\s+Summary\s+Sheet\s*$",
    r"(?i)^Project\s+Summary\s+Sheet\s*$",
    r"(?i)^1\.\s*Summary\s+Sheet\s*$",
    r"(?i)^1.Summary\s+Sheet\s*$",
    r"(?i)^Summary\s+Sheet\s*$",
    r"(?i)^1\.\s*Project\s+Summary\s+Sheet\s*$",
    r"(?i)^I\.\s*Project\s+Summary\s+Sheet\s*$",
    r"(?i)^I\.\s*Summary\s+Sheet\s*$"
    r"(?i)^I. Result-Based Program (RBP) Summary Sheet"
]


stop_patterns = [
   # r"(?i)Strategic\s+Context",
    r"(?i)Team\s+Members",
    r"(?i)Project\s+Team\s+Members",
    r"(?i)Table\s+of\s+Contents",
    r"(?i)Context"
]


start_keyword = (
    r"(?i)^1\.\s*Project\s+Summary\s+Sheet\s*$|"
    r"(?i)^Project\s+Summary\s+Sheet\s*$|"
    r"(?i)^1\.\s*Summary\s+Sheet\s*$|"
    r"(?i)^Summary\s+Sheet\s*$|"
    r"(?i)^I\.\s*Project\s+Summary\s+Sheet\s*$|"
    r"(?i)^I\.\s*Summary\s+Sheet\s*$"
)

end_keyword = (
   # r"(?i)Strategic\s+Context|"
    r"(?i)Team\s+Members|"
    r"(?i)Project\s+Team\s+Members|"
    r"(?i)Table\s+of\s+Contents|"
    r"(?i)Context"
)



In [8]:
pip install langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     -------------------------------------- 981.5/981.5 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993251 sha256=036b568c44ca09784e6f8f424d46e0870140f4c6f3bd6edcb87dd5b4f1c63a5e
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\c1\67\88\e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import re
import fitz  # PyMuPDF for PDF text extraction
import pandas as pd
from langdetect import detect



def identify_table_of_contents(pdf):
    """
    Identifies the Table of Contents (TOC) page based on specific patterns.
    Returns the human-readable page number (1-indexed), or 1 (default to first page) if not found.
    """
    # Iterate over each page in the PDF
    for page_num in range(pdf.page_count):
        page = pdf[page_num]
        page_text = page.get_text("text")
        
        # Regex to match 'Table of Contents' with spaces between characters
        if re.search(r'\b(T\s*a\s*b\s*l\s*e\s*\s*o\s*f\s*\s*C\s*o\s*n\s*t\s*e\s*n\s*t\s*s|Table of Contents|Contents)\b', page_text, re.IGNORECASE):
            return page_num + 1  # Return as human-readable page number
    
    # Default to the first page if no TOC is found
    return 1

def read_pdf(pdf_path, start_patterns, stop_patterns):
    """
    Reads text from a PDF between specified start and stop patterns.
    Searches for start patterns after the TOC page.
    """
    try:
        with fitz.open(pdf_path) as pdf_document:
            # Identify the Table of Contents (TOC) page
            toc_page = identify_table_of_contents(pdf_document)
            if toc_page:
                print(f"TOC identified on page {toc_page}")
            else:
                print("TOC not found. Starting search from page 1.")
                toc_page = 1  # Default to first page if TOC is not found

            # Get the TOC content to exclude
            toc_content = ""
            if toc_page:
                toc_page_obj = pdf_document[toc_page - 1]
                toc_content = toc_page_obj.get_text("text")

            # Initialize variables
            text_content = ""
            start_found = False
            extracted_text = None

            # Search for the start pattern from the page after TOC
            for start_pattern in start_patterns:
                for page_num in range(toc_page + 1, pdf_document.page_count + 1):  # 1-indexed
                    page = pdf_document[page_num - 1]  # Adjust to 0-indexed
                    page_text = page.get_text("text")
                   
                    # Skip if this text is part of TOC
                    if page_text in toc_content:
                        continue

                    start_match = re.search(start_pattern, page_text, re.MULTILINE)
                    if start_match:
                        start_found = True
                        start_index = start_match.end()
                        text_content += page_text[start_index:]  # Append text after start match
                        print(f"Matched start pattern: '{start_pattern}' on page {page_num}")
                        break
                if start_found:
                    break

            if not start_found:
                print("Project Summary Not Found")
                return "Project Summary Not Found"

            # Search for the stop pattern
            for page_num in range(page_num + 1, pdf_document.page_count + 1):  # Continue from start match
                page = pdf_document[page_num - 1]
                page_text = page.get_text("text")
               
                # Skip if this text is part of TOC
                if page_text in toc_content:
                    continue
                   
                text_content += page_text
                for stop_pattern in stop_patterns:
                    stop_match = re.search(stop_pattern, text_content, re.MULTILINE)
                    if stop_match:
                        extracted_text = text_content[:stop_match.start()]  # Text up to stop match
                        print(f"Matched stop pattern: '{stop_pattern}' on page {page_num}")
                        break
                if extracted_text:
                    extracted_text = clean_text(extracted_text, stop_patterns)
                    break

            if not extracted_text:
                print("Stop pattern not found. Returning text after start pattern.")
                return text_content.strip()

            return extracted_text.strip()

    except Exception as e:
        print(f"An error occurred: {e}")
        return "Error processing the PDF"



def extract_section_from_pdf(pdf_path, start_keyword, end_keyword, secondary_start, secondary_end):
    extracted_text = ""
    start_page = None
    found_start_pattern = None
    found_end_pattern = None

    with fitz.open(pdf_path) as pdf:
        toc_page = identify_table_of_contents(pdf)

        # Search Using Secondary Keywords
        for page_num in range(toc_page - 1 if toc_page else pdf.page_count):
            page = pdf[page_num]
            text = page.get_text("text")

            # Check for the secondary start pattern
            if re.search(secondary_start, text, re.IGNORECASE) and start_page is None:
                start_page = page_num + 1
                start_index = re.search(secondary_start, text, re.IGNORECASE).start()
                found_start_pattern = text[start_index:start_index + len(re.search(secondary_start, text, re.IGNORECASE).group())]
                text = text[start_index + len(found_start_pattern):]

            # Only check for the end pattern if we found the start pattern
            if found_start_pattern and re.search(secondary_end, text, re.IGNORECASE):
                end_index = re.search(secondary_end, text, re.IGNORECASE).start()
                found_end_pattern = text[end_index:end_index + len(re.search(secondary_end, text, re.IGNORECASE).group())]
                extracted_text += text[:end_index]
                break
            else:
                if start_page:
                    extracted_text += text

        # If no text is found using secondary keywords, use primary keywords
        if not extracted_text:
            max_pages_to_search = 10  # Limit search to 10 pages
            search_limit = min(toc_page + max_pages_to_search - 1, pdf.page_count) if toc_page else pdf.page_count

            for page_num in range(toc_page, search_limit if toc_page else pdf.page_count):
                page = pdf[page_num]
                text = page.get_text("text")
                
                # Check for the primary start pattern
                if re.search(start_keyword, text, re.IGNORECASE) and start_page is None:
                    start_page = page_num + 1
                    start_index = re.search(start_keyword, text, re.IGNORECASE).start()
                    found_start_pattern = text[start_index:start_index + len(re.search(start_keyword, text, re.IGNORECASE).group())]
                    text = text[start_index + len(found_start_pattern):]

                # Only check for the end pattern if we found the start pattern
                if found_start_pattern and re.search(end_keyword, text, re.IGNORECASE):
                    end_index = re.search(end_keyword, text, re.IGNORECASE).start()
                    found_end_pattern = text[end_index:end_index + len(re.search(end_keyword, text, re.IGNORECASE).group())]
                    extracted_text += text[:end_index]
                    break
                else:
                    if start_page:
                        extracted_text += text

        # If no section was found, return an empty string
        if not extracted_text:
            print(f"Project Summary Table is Missing.\n")
            extracted_text = ""
            return extracted_text
        else:
            # Debugging information
            print(f"Using short method Start Pattern for Project Summary: {found_start_pattern if found_start_pattern else 'Not found'}")
            print(f"End Pattern for Project Summary: {found_end_pattern if found_end_pattern else 'Not found'}\n\n")
            #print(extracted_text.strip())
            return extracted_text.strip()



    

def is_pdf_in_english(text):
    try:
        return detect(text) == 'en'
    except Exception as e:
        print(f"Error detecting language: {e}")
        return False
    

def clean_text(text, cleaning_patterns):
    removed_patterns = []  # To store removed patterns

    for end_marker in cleaning_patterns:
        # Keep removing all occurrences of the end_marker in the text
        while re.search(end_marker, text):
            # Find the position of the first occurrence of the end_marker
            end_marker_index = re.search(end_marker, text)
            if end_marker_index:
                # Print the matched pattern (the removed part)
                print(f"Pattern removed: {end_marker}")
                
                # Truncate the text before the first occurrence of the end_marker (including the pattern itself)
                text = text[:end_marker_index.start()]
                removed_patterns.append(end_marker)  # Add the removed pattern to the list

    return text


def process_pdfs_in_folder(folder_path, start_patterns, stop_patterns, output_excel):
    results = []

    # PROJECT DESCRIPTION
    
    project_description_start = [
    r"Project\s*Description",
    r"The\s*Project",
    r"The\s*Project\s*Description",
    r"Description"
    r"Project Objectives ",
    r"Brief Project Description",
    #SPECIAL CASES:
    r"(?i)^\d+\.\s*Program\s+Description\s*$",
    r"(?i)^\d+\.\s*Project\s+Description\s*$",
    r"(?i)^[IVXLCDM]+\.\s*PROGRAM\s+DESCRIPTION\s*$"


    ]

    project_description_end = [
    r"Project Implementation Period",
    r"Implementation Period",
    r"(?i)^\d+\.\s*Description\s+and\s+Components\s*$",
    r"(?i)^\d+\.\s*Project\s+Description\s+and\s+Components\s*$"
    r"Project\s*Description\s*and\s*Components",
    r"(?i)^\d+\.\s*Components",
    r"(?i)^[C]\.\s*Components\s*$",
    r"(?i)^[IVXLCDM]+\.\s*ASSESMENT\s+SUMMARY\s*$",
    r"(?i)^[A-Z]\.\s*Description,\s*Component,\s*Cost\s+and\s+Financing\s+Plan\s*$",
    r"(?i)^Table\s+\d+\.\s*$"


    
    ]


    # Define Project Components patterns
    project_component_start = [
        r"C\s*\.\s{1}P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}\s{1}C\s{1}o\s{1}m\s{1}p\s{1}o\s{1}n\s{1}e\s{1}n\s{1}t\s{1}s",  # Handles spacing variations in "C. PROJECT COMPONENTS"
        r"P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}\s{1}C\s{1}o\s{1}m\s{1}p\s{1}o\s{1}n\s{1}e\s{1}n\s{1}t\s{1}s",  # Handles spacing variations in "PROJECT COMPONENTS"
       
       #-----
        r"(?i)^[C]\.\s*Project\s+Description\s+and\s+Components\s*$", # Can replace C with A-Z if needed
        r"(?i)^[C]\.\s*Description\s+and\s+Components\s*$",
        r"(?i)^[C]\.\s*Components\s*$",
        r"(?i)^[A-Z]\.\s*Project\s+Design\s+and\s+Components\s*$",
       #-----

        r"[ABCD]\.\s*Project\s*Components",  # "A. Project Components"
        r"[ABCD]\.\s*(Project|Program)\s*Components",  # Covers all variants with A/B/C/D
        # r"\d\.\d?\s*Project\s*Components\s*and\s*Expected\s*Outputs",
        r"\d\.\d?\s*Project\s*Components\s*and",
        r"\d\.\d?\s*Project\s*Components",  # "2.1 Project Components"
        r"\d\.\d\s*(Project|Program|Programme|Programm)\s*Components",  # Double numbered sections, eg. 1.1 Project Components
        r"C\s*\.\s*Project\s*Components",  # "C. Project Components"
        r"[ABCD]\.\s*(Project|Programme|Programm|Program)\s*Components",  # Variations for A/B/C/D with Components
          # "Components:" followed by any characters
        r"Component\s*1\s*",  # "Component 1" followed by any characters
        r"(Component\s*\d+\.)",  # Matches "Component 1." "Component 2." etc.
        r"(Component\s*\d+:\s*)",  # Matches "Component 1: " etc.
        r"(Component\s*\d+)",  # Matches "Component 1", "Component 2", etc.
        r"Component\s+One\s+",  # "Component One" followed by any characters
        r"Component\s*Name[\s\S]",  # "Component Name" followed by any characters
        r"Component\s*1[\s\S]",
        r"following\s*components\.",  # "following components."
        r"Key\s*components",  # "Key components"
        r"Components\s+are",  # "Components are"
        r"main\s*components",
        r"components:",  
        r"components\s*:",  # "components:" followed by a colon
        r"Project\s*Component|Project\s*/\s*Component|Project\s*/\s*Components",  # Variations for slashes
        r"Component\s+A\s+",  # "Component A" followed by any characters
        r"Component\s+I\s+",
        r"components,"
    ]

    
    project_component_end = [
        
    # Matches "P R O J E C T C O S T A N D F I N A N C I N G" with one space between each character
    r"(?i)P\s{1}R\s{1}O\s{1}J\s{1}E\s{1}C\s{1}T\s{1}\s{1}C\s{1}O\s{1}S\s{1}T\s{1}\s{1}A\s{1}N\s{1}D\s{1}\s{1}F\s{1}I\s{1}N\s{1}A\s{1}N\s{1}C\s{1}I\s{1}N\s{1}G",

    # Matches various forms of "COST ESTIMATES AND FINANCING PLAN"
    r"(?i)COST\s*ESTIMATES\s*AND\s*FINANCING\s*PLAN",
    r"(?i)ESTIMATED\s*COSTS\s*AND\s*FINANCING\s*PLAN",
    r"(?i)Project\s*Cost\s*and\s*Financing\s*Arrangements",

    # Matches various forms of "Cost and Financing Plan"
    r"(?i)Cost\s*and\s*Financing\s*Plan",
    r"(?i)Cost\s*and\s*Financing",
    r"(?i)Project\s*Cost\s*and\s*Finance",
    r"(?i)Project\s*Cost\s*and\s*Financing\s*Plan",
    r"(?i)Project\s*Cost\s*and\s*Finance\s*Plan",

    # Matches "COST AND FINANCING PLAN" and variations
    r"(?i)COST AND FINANCING PLAN",
    r"(?i)COST AND FINANCING",
    r"(?i)PROJECT COST AND FINANCE",
    r"(?i)Project Cost and finance plan",
    r"(?i)PROJECT COST AND FINANCING PLAN",
        
    ]
    combined_end_patterns = project_component_end 

    for filename in os.listdir(folder_path):
        if filename.endswith('.pdf'):
            file_path = os.path.join(folder_path, filename)
            print(f"\n\n\n\n\n\nProcessing file: {filename}")
            try:
                pdf_document = fitz.open(file_path)

                # Extract all text for language check
                text = ""
                for page_num in range(pdf_document.page_count):
                    text += pdf_document[page_num].get_text("text")

                # Check language first
                if not is_pdf_in_english(text):
                    results.append([filename, "Not Supported", "Not Supported"])
                    pdf_document.close()
                    continue

                # Identify the Table of Contents page
                toc_page = identify_table_of_contents(pdf_document)
                if toc_page is not None:
                    print(f"Table of Contents identified on page: {toc_page + 1}")
                    skip_pages = list(range(toc_page + 2))  # Skip TOC page and all pages before it
                else:
                    skip_pages = []

                # Extract text excluding TOC pages (i.e., start from the page after the TOC)
                text = ""
                for page_num in range(pdf_document.page_count):
                    if page_num in skip_pages:  # Skip TOC pages and earlier ones
                        continue
                    text += pdf_document[page_num].get_text("text")
                pdf_document.close()

                # Initialize variable to store the extracted component text
                component_text = None

                # Extract Project Components Section
                # Extract Project Components Section
                for start_pattern in project_component_start:
                    match_start = re.search(start_pattern, text, re.IGNORECASE)
                    if match_start:
                        print(f"Component start: {start_pattern}")
                        start_pos = match_start.end()  # Start extraction after the matched pattern

                        # Use the first end pattern that matches
                        for end_pattern in combined_end_patterns:
                            match_end = re.search(end_pattern, text[start_pos:], re.IGNORECASE)
                            if match_end:
                                print(f"Component end: {end_pattern}")
                                end_pos = match_end.start() + start_pos
                                component_text = text[start_pos:end_pos]
                                component_text = clean_text(component_text, combined_end_patterns)
                                break  # Stop after extracting the first valid component

                        if component_text:  # Break the outer loop once a valid section is found
                            break

                # Fallback if no component text is found
                if not component_text:
                    component_text = "Project Components not found"

                # Add this new block here -> PROJECT DESCRIPTION
                # Initialize variable to store the extracted project description text
                description_text = None

                # Extract Project Description Section
                for start_pattern in project_description_start:
                    match_start = re.search(start_pattern, text, re.IGNORECASE)
                    if match_start:
                        print(f"Project Description start: {start_pattern}")
                        start_pos = match_start.end()  # Start extraction after the matched pattern

                        # Use the first end pattern that matches
                        for end_pattern in project_description_end:
                            match_end = re.search(end_pattern, text[start_pos:], re.IGNORECASE)
                            if match_end:
                                print(f"Project Description end: {end_pattern}")
                                end_pos = match_end.start() + start_pos
                                description_text = text[start_pos:end_pos]
                                description_text = clean_text(description_text, project_description_end)
                                break  # Stop after extracting the first valid description

                        if description_text:  # Break the outer loop once a valid section is found
                            break

                # Fallback if no description text is found
                if not description_text:
                    description_text = "Project Description not found"


                extracted_text = extract_section_from_pdf(file_path, start_keyword, end_keyword, start_keyword, end_keyword)
                
                if not extracted_text:
                    extracted_text = read_pdf(file_path, start_patterns, stop_patterns)
                    if extracted_text is None:
                        extracted_text = "Pattern not found"
                
                results.append([filename, extracted_text, component_text, description_text])


            except Exception as e:
                results.append([filename, f"Error processing file: {e}", "Not Available"])

    df = pd.DataFrame(results, columns=["Filename", "Project Summary", "Project Component", "Project Description"])
    df.to_excel(output_excel, index=False)
    print(f"\n\nResults saved to {output_excel}")

folder_path = r"C:\Users\iamsy\Desktop\work\Asian Infrastructure Investment Bank"
output_excel = r"C:\Users\iamsy\Desktop\work\Final.xlsx"
process_pdfs_in_folder(folder_path, start_patterns, stop_patterns, output_excel)








Processing file: 000162-PAK_Karachi-BRT-Project_President-Approval.pdf
Table of Contents identified on page: 4
Component start: [ABCD]\.\s*Project\s*Components
Component start: [ABCD]\.\s*(Project|Program)\s*Components
Component start: [ABCD]\.\s*(Project|Programme|Programm|Program)\s*Components
Project Description start: Project\s*Description
Project Description end: Project Implementation Period
Pattern removed: Implementation Period
Project Summary Table is Missing.

TOC identified on page 3
Matched start pattern: '(?i)^1\.\s*Summary\s+Sheet\s*$' on page 4
Matched stop pattern: '(?i)Team\s+Members' on page 5






Processing file: 20161213051938915.pdf
Table of Contents identified on page: 4
Project Description start: Project\s*Description
Project Description end: Implementation Period
Project Summary Table is Missing.

TOC identified on page 3
Matched start pattern: '(?i)^1\.\s*Project\s+Summary\s+Sheet\s*$' on page 4
Matched stop pattern: '(?i)Team\s+Members' on page 6







In [29]:
import re

text = """9.

word Components
"""

# Match the pattern for "1." followed by "INTRODUCTION"
#pattern = r"(?i)((\b\d{1,2}\b(\.)?\d{0,1})|\b([ABCD])\b)\.?(\s*)(Project|Program|Programm|Programme)?\s*Components([\s\S]*)"


pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([ABCD])\b)\.?(\s*)(\W)?\s*(\b\w+\b\s*){0,4}Components\s*$([\s\S]*)"

match = re.search(pattern, text, re.M)

if match:
    print("Match found!")
else:
    print("No match found.")


Match found!


In [36]:
import re

pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([ABCD])\b)\.?(\s*)(\W)?\s*(\b\w+\b\s*){0,3}Components(\s*(\b\w+\b\s*){0,3})?\s*$"

# Sample text with different scenarios
texts = [
    "1. Project Components",
    "A. Program Components",
    "1.1 Description Components that should work",
    "B. Section Components",
    "A. Program  and and Components that work wellv"
]

for text in texts:
    match = re.match(pattern, text)
    if match:
        print("Match found:", match.group(0))
    else:
        print("No match for:", text)


Match found: 1. Project Components
Match found: A. Program Components
Match found: 1.1 Description Components that should work
Match found: B. Section Components
Match found: A. Program  and and Components that work wellv


In [86]:
import re
import fitz  # PyMuPDF


# Define the regex pattern
pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([ABCD])\b)\.?\n?(\s*)(\W)?\s*((\b\w+\b\s*){0,1}Components(\s*(\b\w+\b\s*){0,5})?|((\b\w+\b\s*){0,2}Components(\s*(\b\w+\b\s*){0,4})?)|((\b\w+\b\s*){0,3}Components(\s*(\b\w+\b\s*){0,3})?)|((\b\w+\b\s*){0,4}Components(\s*(\b\w+\b\s*){0,2})?)|((\b\w+\b\s*){0,5}Components(\s*(\b\w+\b\s*){0,1})?))\s*$([\s\S]*)"
clean_pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([BCDEFG])\b)\.?\n?Technical Solutions Adopted and Alternatives Considered"
# Sample text with different scenarios

# Sample function to clean text using the regex patterns
def clean_text(text, cleaning_patterns):
    for pattern in cleaning_patterns:
        # Search for the pattern in the text
        match = re.search(pattern, text)
        if match:
            # Print the pattern that is used to truncate the text
            print(f"Pattern used to truncate text: {pattern}")
            
            # Truncate the text after the matched pattern
            text = text[:match.start()]
    
    return text

# Path to the PDF file
pdf_path = "/Users/adilqasin/Documents/WBG/Scrapping/AFDB/Debugga 2/ghana_-_cocoa_sector_institutional_support_project_cosisp_-_project_appraisal_report.pdf"

# Open the PDF document
with fitz.open(pdf_path) as pdf_document:
    # Initialize an empty string to hold all text from all pages
    all_text = ""
    
    # Loop through all pages in the PDF from the last to the first
    for page_num in range(pdf_document.page_count - 1, -1, -1):  # Start from last page
        page = pdf_document.load_page(page_num)
        all_text += page.get_text("text")  # Extract text from each page
    
    print(all_text)
    # Search for the main pattern in the text (from the end of the document)
    match = re.search(pattern, all_text, re.DOTALL)

    if match:
        components_text = match.group(0).strip()
        print(f"Match found with pattern: {pattern}\n\n")
        
        # Clean the extracted text using the clean pattern
        components_text = clean_text(components_text, [clean_pattern])
        print(components_text)

 
IV 
 
ANNEX III - MAP OF GHANA 
 
 
 
 
III 
 
ANNEX II: TABLE OF AFDB PORTFOLIO IN THE COUNTRY 
 
Status of Ongoing AfDB Portfolio in Ghana (UA Million)  
(August 2018) 
Project Name 
Loan/Grant 
Amount in UA 
Million 
Approval Date 
Disbursement 
Rate 
Closing 
Date 
PUBLIC SECTOR PROJECTS 
 
 
 
 
1. Accra Urban Transport Project 
60.00 
9/28/2016 
         12.6% 
12/31/2020 
2. Electricity Distribution System 
Reinforcement 
48.46 
02/26/2014 
23% 
10/30/2019 
3. Renewable Mini-Grids & Solar Stand 
Alone Systems 
0.65 
06/04/2015 
9.17% 
06/30/2019 
4. Net Metered Solar PV for SMEs & 
lighting 
0.44                  
06/16/2015 
27.27% 
06/30/2019 
5. Sogakope-Lome Water Transfer 
Project 
 
1.13 
12/18/2013 
22.2% 
 
12/31/2018 
6. Greater Accra Sustainable Sanitation 
and Livelihoods Improvement Project 
35.95 
03/29/2017 
0.54% 
03/31/2022 
7. Rural Enterprises Project III  
49.69 
12/19/2012 
27% 
09/30/2020 
8. Engaging Local Communities in 
REDD++ 
13.20 
01/22/2014 
57% 
1

In [88]:
import fitz  # PyMuPDF
import re

# Define the patterns to extract text between
start_pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([ABCD])\b)\.?\n?(\s*)(\W)?\s*((\b\w+\b\s*){0,1}Components(\s*(\b\w+\b\s*){0,5})?|((\b\w+\b\s*){0,2}Components(\s*(\b\w+\b\s*){0,4})?)|((\b\w+\b\s*){0,3}Components(\s*(\b\w+\b\s*){0,3})?)|((\b\w+\b\s*){0,4}Components(\s*(\b\w+\b\s*){0,2})?)|((\b\w+\b\s*){0,5}Components(\s*(\b\w+\b\s*){0,1})?))\s*$"
end_pattern = r"(?i)((\b\d{1}\b(\.)?\d{0,1})|\b([BCDEFG])\b)\.?\n?Technical\s*Solutions"


# Sample tex
# Open the PDF file
# Path to the PDF file

pdf_path = "/Users/adilqasin/Documents/WBG/Scrapping/AFDB/Debugga 2/ghana_-_cocoa_sector_institutional_support_project_cosisp_-_project_appraisal_report.pdf"


# Open the PDF file
pdf_file = fitz.open(pdf_path)

# Function to extract text between two patterns
def extract_text_between_patterns(pdf_file, start_pattern, end_pattern, skip_pages=5):
    text = ""
    
    # Loop through the pages, starting from page 6 (index 5) to skip the first 5 pages
    for page_num in range(skip_pages, len(pdf_file)):
        page = pdf_file.load_page(page_num)  # Load the page
        page_text = page.get_text("text")  # Extract text from the page
        
        # Check if the start pattern exists and extract text until the end pattern is found
        match_start = re.search(start_pattern, page_text)
        match_end = re.search(end_pattern, page_text)
        
        if match_start:
            text_started = False
            for line in page_text.splitlines():
                if match_start.group(0) in line:
                    text_started = True  # Start extracting after this line
                if text_started:
                    text += line + "\n"
                if match_end and match_end.group(0) in line and text_started:
                    break  # Stop extracting when the end pattern is found
    
    return text

# Extract text between patterns
extracted_text = extract_text_between_patterns(pdf_file, start_pattern, end_pattern)

# Print the extracted text
print(extracted_text)

# Close the PDF file
pdf_file.close()


In [109]:
import re
import fitz  # PyMuPDF

# Function to extract text between two patterns from a PDF
def extract_text_between_patterns(pdf_file, start_pattern, end_pattern):
    try:
        # Open the PDF file
        doc = fitz.open(pdf_file)
        
        # Initialize an empty string to store the extracted text
        extracted_text = ''
        
        # Iterate over all pages in the PDF
        for page_num in range(doc.page_count):
            page = doc.load_page(page_num)
            
            # Extract text from the page
            page_text = page.get_text("text")
            
            # Combine flags at the start of the regex pattern
            regex_pattern = f'(?si)({start_pattern})(.*?){end_pattern}'  # (?si) for case-insensitive and dot-matching newlines
            matches = re.findall(regex_pattern, page_text)  # (?s) allows dot to match newline

            # Append matched text (everything between the start and end patterns)
            for match in matches:
                extracted_text += match[1].strip() + '\n'

        return extracted_text
    except Exception as e:
        print(f"Error extracting text: {e}")
        return None

start_pattern = r"\b([A-D](\.|-)?|(\d(\.\d{1,3}){0,2})(\.|-)?)\b\s*(\W)?\s*\n?((\b[A-Za-z0-9]+\b\s*){0,5}Components(\s*(\b[A-Za-z0-9]+\b\s*){0,5})?)\s*$"
#start_pattern = r"\b([A-D](\.|-)?|(\d(\.\d{1,3}){0,2})(\.|-)?)\b\s*(\W)?\s*\n?((\b\w+\b\s*){0,1}Components(\s*(\b\w+\b\s*){0,5})?|((\b\w+\b\s*){0,2}Components(\s*(\b\w+\b\s*){0,4})?)|((\b\w+\b\s*){0,3}Components(\s*(\b\w+\b\s*){0,3})?)|((\b\w+\b\s*){0,4}Components(\s*(\b\w+\b\s*){0,2})?)|((\b\w+\b\s*){0,5}Components(\s*(\b\w+\b\s*){0,1})?))\s*$"
end_pattern = r"\b([A-D](\.|-)?|(\d(\.\d{1,3}){0,2})(\.|-)?)\b\s*(\W)?\s*\n?Technical\s*Solutions"

# Path to your PDF file
pdf_file = "/Users/adilqasin/Documents/WBG/Scrapping/AFDB/Debugga 2/ghana_-_cocoa_sector_institutional_support_project_cosisp_-_project_appraisal_report.pdf"

# Extract the text between the two patterns
text_between_patterns = extract_text_between_patterns(pdf_file, start_pattern, end_pattern)

if text_between_patterns:
    # Print or save the extracted text
    print(text_between_patterns)
else:
    print("No matching text found.")


No matching text found.
